# Rumelhart 1986 reproduction: family trees

A reproduction of the family-tree relational-learning experiment from Rumelhart, Hinton & Williams (1986) / Hinton (1986).

**The task**: two isomorphic family trees (English and Italian, 24 people, 12 relation types) give 104 `(person1, relation, person2)` facts. A network sees `person1` and `relation` as one-hot inputs and must output `person2` — not by lookup, but by inferring family structure from raw, arbitrary symbols. Four facts are held out entirely (`TEST_TRIPLES`); a network that only memorized the training set gets them wrong, one that actually learned the relational structure gets them right.

**What counts as "right" turned out to be the whole story here** — see the results section below.

This notebook is a narrative walkthrough with the key figures embedded. For the full design rationale and every technique tried, see [`TECHNIQUES.md`](../TECHNIQUES.md). For the complete chronological experiment log (including every dead end), see [`FINDINGS.md`](../FINDINGS.md).

## Architecture

`TreeNet` ([`model/network.py`](../model/network.py)) follows Figure 3 of the paper: `person1` and `relation` are each compressed by their own encoding layer, concatenated, passed through one hidden layer, and expanded back into a 24-unit output over all people.

Two deliberate departures from the paper's literal spec (both empirically load-bearing, not arbitrary — see `TECHNIQUES.md` §3):

- **Asymmetric encoding split**: person gets 1 dimension, relation gets 5 (the paper uses an even 6/6). This asymmetric bottleneck is what let the person encoding organize cleanly by generation and nationality — an even split left too much room to just memorize identity.
- **Sigmoid encoding layers**: keeps the network a uniform sigmoid chain end to end, rather than the paper's implied linear pass-through.

## Key techniques

The training recipe (SGD + Grokfast + delayed weight decay) is the result of an extensive search — full catalog in [`TECHNIQUES.md`](../TECHNIQUES.md). The headline ones:

- **Weight decay as a *generalization* mechanism, not just regularization** (§4.1) — this is a textbook case for *grokking* (Power et al., 2022): the network memorizes first, then, much later, decay erodes a high-norm "memorizing" solution in favor of a lower-norm "generalizing" one that fits the same training data.
- **Grokfast** (§4.3, Lee et al. 2024) — amplifies the slow-varying component of the gradient, meaningfully accelerating that memorize-to-generalize transition.
- **The person/relation capacity bottleneck** (§4.9) — nine separate experiments giving the network *more* capacity (wider layers, depth, a bilinear interaction, different init) all made test performance *worse*. Capacity was never the bottleneck here.
- **Argmax-in-valid-set evaluation** (§4.15) — the most consequential finding of the whole project: an absolute-confidence threshold and a relative-ranking criterion can disagree completely on the same trained model. See the results section below.

## What the network actually learned

### Person encoding

The network was never told which people are English vs. Italian, or what generation they belong to — both emerge spontaneously from a single learned scalar per person:

![Person encoding](../figures/person_encoding.png)

### Held-out query results

For each held-out test query, here are the model's top-3 output activations. The dashed line is the paper's 0.8 pass threshold:

![Test results](../figures/test_results.png)

The model's top-2 picks are, every time, the two *actual* correct answers (e.g. Colin genuinely has two uncles, Arthur and Charles) — it isn't confused between right and wrong, it just favors whichever answer it saw directly trained over the one it had to infer via sibling-transfer.

### Regenerating these figures

The cell below retrains the model from scratch (deterministic at `SEED=42`, ~5-6 minutes) and regenerates both PNGs above. Not required to read this notebook — only run it if you want to reproduce the figures yourself.

In [ ]:
import sys
sys.path.insert(0, "..")
import visualize

visualize.main()

## Final results

| Grading criterion | `TEST_TRIPLES` (multi-answer) | `VAL_TRIPLES` (single-answer) |
|---|---|---|
| Paper-literal (output > 0.8) | 0/4 | 0/4 |
| Argmax-in-valid-set (matches how the field's benchmark numbers are graded) | **4/4** | **4/4** |

Beats the researched external benchmark for this exact reproduction (1.9/4 average, 3/4 best, across two independent reproductions) under the grading convention those numbers actually use. The honest gap that remains: the network reliably *knows* the right answer relative to wrong ones, but hasn't learned to be fully confident about answers it only ever saw demonstrated for a person's sibling, not that person directly — see `TECHNIQUES.md` §4.12 and §5 for the full mechanism and what was tried (and didn't work) to close it.

## Further reading

- [`TECHNIQUES.md`](../TECHNIQUES.md) — design decisions, every technique tried (including what failed and why), and takeaways for future ML work.
- [`FINDINGS.md`](../FINDINGS.md) — the complete chronological experiment log.
- [`train.py`](../train.py) / [`model/network.py`](../model/network.py) — the finalized, clean implementation.